In [18]:
import ast

# Function to convert code string to AST
def code_to_ast(code_string):
	try:
		return ast.parse(code_string)
	except SyntaxError:
		return None # some code snippets are not valid (python 2 instead of python 3)

In [19]:
# Linearizing the AST (by passing first node) into a list of tokens
def linearize_ast(node, tokens):
	if node is None:
			return

	# 1. Aggiungiamo sempre il tipo di nodo (es. FunctionDef, arg, ecc.)
	node_type = type(node).__name__
	tokens.append(node_type)

	# 2. Aggiungiamo il valore SPECIFICO solo per i nodi che ci interessano
	if isinstance(node, ast.FunctionDef):
			tokens.append(f"FUNC_{node.name}")
	elif isinstance(node, ast.arg):
			tokens.append(f"ARG_{node.arg}")
	elif isinstance(node, ast.Name):
		# Opzionale: aggiungi i nomi delle variabili se vuoi più ROUGE
		tokens.append(f"VAR_{node.id}")

	# 3. Ricorsione sui figli
	for child in ast.iter_child_nodes(node):
		linearize_ast(child, tokens)

In [20]:
linearized_tree = []
linearize_ast(code_to_ast("def add(a, b): return a + b + 3"), linearized_tree)
print(linearized_tree)

['Module', 'FunctionDef', 'FUNC_add', 'arguments', 'arg', 'ARG_a', 'arg', 'ARG_b', 'Return', 'BinOp', 'BinOp', 'Name', 'VAR_a', 'Load', 'Add', 'Name', 'VAR_b', 'Load', 'Add', 'Constant']


Let's now build a dictionary with Corpora, exactly as we have done before.

In [21]:
from datasets import load_dataset

train_dataset = load_dataset(
	"json",
	data_files = "./../../data/raw/dataset/python/train.jsonl",
	split = "train")
valid_dataset = load_dataset(
	"json",
	data_files = "./../../data/raw/dataset/python/valid.jsonl",
	split = "train")
test_dataset = load_dataset(
	"json",
	data_files = "./../../data/raw/dataset/python/test.jsonl",
	split = "train")

In [22]:
train_linearized_trees = []
train_valid_indices = []
for i, code in enumerate(train_dataset['code']):
	linearized_tree = []
	tree = code_to_ast(code)
	if(tree is not None):
		linearize_ast(tree, linearized_tree)
		train_linearized_trees.append(linearized_tree)
		train_valid_indices.append(i)

valid_linearized_trees = []
valid_valid_indices = []
for i, code in enumerate(valid_dataset['code']):
	linearized_tree = []
	tree = code_to_ast(code)
	if(tree is not None):
		linearize_ast(tree, linearized_tree)
		valid_linearized_trees.append(linearized_tree)
		valid_valid_indices.append(i)

test_linearized_trees = []
test_valid_indices = []
for i, code in enumerate(test_dataset['code']):
	linearized_tree = []
	tree = code_to_ast(code)
	if(tree is not None):
		linearize_ast(tree, linearized_tree)
		test_linearized_trees.append(linearized_tree)
		test_valid_indices.append(i)

print(train_linearized_trees[0])

['Module', 'FunctionDef', 'FUNC_split_phylogeny', 'arguments', 'arg', 'ARG_p', 'arg', 'ARG_level', 'Constant', 'Expr', 'Constant', 'Assign', 'Name', 'VAR_level', 'Store', 'BinOp', 'Name', 'VAR_level', 'Load', 'Add', 'Constant', 'Assign', 'Name', 'VAR_result', 'Store', 'Call', 'Attribute', 'Name', 'VAR_p', 'Load', 'Load', 'Name', 'VAR_level', 'Load', 'Return', 'BinOp', 'BinOp', 'Subscript', 'Name', 'VAR_result', 'Load', 'Constant', 'Load', 'Add', 'Name', 'VAR_level', 'Load', 'Add', 'Subscript', 'Call', 'Attribute', 'Subscript', 'Name', 'VAR_result', 'Load', 'Constant', 'Load', 'Load', 'Constant', 'Constant', 'Load']


In [23]:
from gensim import corpora
code_dictionary = corpora.Dictionary(train_linearized_trees)
special_tokens = {'[UNK]': 0, '[PAD]': 1, '[BOS]': 2, '[EOS]': 3}
code_dictionary.patch_with_special_tokens(special_tokens)

In [24]:
code_dictionary.token2id

{'ARG_level': 615374,
 'ARG_p': 615375,
 'Add': 615376,
 'Assign': 615377,
 'Attribute': 4,
 'BinOp': 5,
 'Call': 6,
 'Constant': 7,
 'Expr': 8,
 'FUNC_split_phylogeny': 9,
 'FunctionDef': 10,
 'Load': 11,
 'Module': 12,
 'Name': 13,
 'Return': 14,
 'Store': 15,
 'Subscript': 16,
 'VAR_level': 17,
 'VAR_p': 18,
 'VAR_result': 19,
 'arg': 20,
 'arguments': 21,
 'ARG_d': 22,
 'Compare': 23,
 'Eq': 24,
 'ExceptHandler': 25,
 'FUNC_ensure_dir': 26,
 'If': 27,
 'Not': 28,
 'Try': 29,
 'UnaryOp': 30,
 'VAR_OSError': 31,
 'VAR_d': 32,
 'VAR_errno': 33,
 'VAR_msg': 34,
 'VAR_oe': 35,
 'VAR_os': 36,
 'VAR_twdd': 37,
 'ARG_fnh': 38,
 'ARG_mode': 39,
 'FUNC_file_handle': 40,
 'Raise': 41,
 'VAR_ValueError': 42,
 'VAR_file': 43,
 'VAR_fnh': 44,
 'VAR_handle': 45,
 'VAR_isinstance': 46,
 'VAR_mode': 47,
 'VAR_open': 48,
 'VAR_str': 49,
 'ARG_categories': 50,
 'ARG_header': 51,
 'ARG_imap': 52,
 'And': 53,
 'Assert': 54,
 'BoolOp': 55,
 'Continue': 56,
 'Dict': 57,
 'FUNC_gather_categories': 58,
 'F

In [25]:
len(code_dictionary)

615378

We can see that we went from a code_dictionary of over 1 million of tokens (see notebook 02) to just over 400k. This can help speed up the training and give some more accurate results. So, let's try this by just copy-pasting the model from notebook 03. We're gonna use the same vocab for docstrings. We're gonna save the processed dataset and test it on the next notebook.

In [27]:
from datasets import load_from_disk
old_dataset = load_from_disk("../../data/processed/notebooks/tokenized_codexglue")

In [28]:
train_input_ids = [
	[code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in tree]
	for tree in train_linearized_trees
]

valid_input_ids = [
	[code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in tree]
	for tree in valid_linearized_trees
]

test_input_ids = [
	[code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in tree]
	for tree in test_linearized_trees
]

In [29]:
from datasets import Dataset, DatasetDict

processed_datasets = DatasetDict({
	'train': Dataset.from_dict({
		'input_ids': train_input_ids,
		'labels': [old_dataset['train']['labels'][i] for i in train_valid_indices]
	}),
	'valid': Dataset.from_dict({
		'input_ids': valid_input_ids,
		'labels': [old_dataset['valid']['labels'][i] for i in valid_valid_indices]
	}),
	'test': Dataset.from_dict({
		'input_ids': test_input_ids,
		'labels': [old_dataset['test']['labels'][i] for i in test_valid_indices]
	})
})

We're gonna save a copy of our docstring dictionary, so as to have everything organized.

In [30]:
docstring_dictionary = corpora.Dictionary.load('./../../data/processed/notebooks/tokenized_codexglue/docstring_dictionary.pt')

In [31]:
processed_datasets.save_to_disk('./../../data/processed/notebooks/ast/')
code_dictionary.save('./../../data/processed/notebooks/ast/code_dictionary.pt')
docstring_dictionary.save('./../../data/processed/notebooks/ast/docstring_dictionary.pt')

Saving the dataset (1/1 shards): 100%|██████████| 14761/14761 [00:00<00:00, 775044.71 examples/s]
